In [1]:
import os
import scipy.io

chair_dir = './dropbox/Chair/'
model_idx = 1  # change this to the model you want

ops_path = os.path.join(chair_dir, 'ops', f'{model_idx}.mat')
ops_mat = scipy.io.loadmat(ops_path)

print("Keys in ops mat file:", ops_mat.keys())

Keys in ops mat file: dict_keys(['__header__', '__version__', '__globals__', 'op'])


In [2]:
ops_data = ops_mat['op'][0]  # get the 1D array of node types
print("Node types (ops_data):", ops_data)
print("Number of nodes:", len(ops_data))

Node types (ops_data): [0 0 1 0 1]
Number of nodes: 5


In [3]:
labels_path = os.path.join(chair_dir, 'labels', f'{model_idx}.mat')
labels_mat = scipy.io.loadmat(labels_path)

print("Keys in labels mat file:", labels_mat.keys())

Keys in labels mat file: dict_keys(['__header__', '__version__', '__globals__', 'label'])


In [4]:
labels_data = labels_mat['label'][0]  # 1D array of leaf node labels
print("Leaf node labels:", labels_data)
print("Number of leaf nodes:", len(labels_data))

Leaf node labels: [1 2 0]
Number of leaf nodes: 3


In [5]:
boxes_path = os.path.join(chair_dir, 'boxes', f'{model_idx}.mat')
boxes_mat = scipy.io.loadmat(boxes_path)

print("Keys in boxes mat file:", boxes_mat.keys())

Keys in boxes mat file: dict_keys(['__header__', '__version__', '__globals__', 'box'])


In [6]:
boxes_data = boxes_mat['box']
print("Boxes shape:", boxes_data.shape)
print("First box:", boxes_data[0])

Boxes shape: (12, 3)
First box: [0.0030865  0.00419755 0.0302842 ]


In [7]:
part_mesh_path = os.path.join(chair_dir, 'part mesh indices', f'{model_idx}.mat')
part_mesh_mat = scipy.io.loadmat(part_mesh_path)

print("Keys in part mesh indices mat file:", part_mesh_mat.keys())

Keys in part mesh indices mat file: dict_keys(['__header__', '__version__', '__globals__', 'cell_boxs_correspond_objSerialNumber', 'shapename'])


In [8]:
mesh_indices_data = part_mesh_mat['cell_boxs_correspond_objSerialNumber'][0]
partnet_id = int(part_mesh_mat['shapename'][0])

for idx, leaf in enumerate(mesh_indices_data):
    print(f"Leaf node {idx} mesh indices:", leaf[0])

print("PartNet ID:", partnet_id)

Leaf node 0 mesh indices: [5 6 7 8]
Leaf node 1 mesh indices: [ 9 10]
Leaf node 2 mesh indices: [1 2 3 4]
PartNet ID: 1095


In [9]:
import numpy as np

obb_file = os.path.join(chair_dir, 'obbs', f'{partnet_id}.obb')

def read_obb_file(filepath):
    obbs = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            # skip header and metadata lines
            if len(line) == 0 or line.startswith(('N','C','S','L')):
                continue
            vals = line.split()
            if len(vals) < 12:
                continue
            arr = np.array([float(v) for v in vals[:12]], dtype=np.float32)
            obbs.append(arr)
    return obbs

genshape = read_obb_file(obb_file)
print(f"Read {len(genshape)} OBBs")
print("First OBB:", genshape[0])

Read 3 OBBs
First OBB: [ 0.0302842   0.488309   -0.315397   -0.995024    0.0971271  -0.0222212
 -0.00601635  0.164046    0.986434    0.0994548   0.981659   -0.162646  ]


In [10]:
from draw3dobb import draw
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def save_obb_plot(genshape, labels, filename='chair_obb.png'):
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(-0.7, 0.7)
    ax.set_zlim(-0.7, 0.7)
    
    # Color map for semantic labels: 0-back, 1-seat, 2-leg, 3-armrest
    colors_map = {0: 'red', 1: 'green', 2: 'blue', 3: 'yellow'}
    
    for i, obb in enumerate(genshape):
        color = colors_map.get(labels[i], 'black')
        draw(ax, obb, color)
    
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"3D OBB plot saved to {filename}")

# Call the function with leaf node labels
save_obb_plot(genshape, labels_data, 'chair_1095_obb.png')

3D OBB plot saved to chair_1095_obb.png


In [11]:
import k3d
import numpy as np

def make_box_k3d(obb, color=0xff0000, opacity=0.3):
    """
    Convert a 12-element OBB vector into a K3D mesh.
    obb: np.array of shape (12,) -> [center(3), lengths(3), dir1(3), dir2(3)]
    """
    c = obb[0:3]
    l = obb[3:6]
    d1 = obb[6:9]; d1 /= np.linalg.norm(d1)
    d2 = obb[9:12]; d2 /= np.linalg.norm(d2)
    d3 = np.cross(d1, d2); d3 /= np.linalg.norm(d3)
    d1 *= l[0]/2
    d2 *= l[1]/2
    d3 *= l[2]/2

    # compute 8 corners
    corners = np.array([
        c - d1 - d2 - d3,
        c - d1 + d2 - d3,
        c + d1 - d2 - d3,
        c + d1 + d2 - d3,
        c - d1 - d2 + d3,
        c - d1 + d2 + d3,
        c + d1 - d2 + d3,
        c + d1 + d2 + d3,
    ], dtype=np.float32)

    # define triangles for cube faces
    indices = np.array([
        0,1,2, 0,2,3,
        4,5,6, 4,6,7,
        0,1,5, 0,5,4,
        2,3,7, 2,7,6,
        1,2,6, 1,6,5,
        0,3,7, 0,7,4
    ], dtype=np.uint32)

    return k3d.mesh(corners, indices, color=color, opacity=opacity)

# -----------------------------
# Example usage with your parsed OBBs
# -----------------------------
# genshape = read_obb_file(...)  # your previous robust parser
# labels_data = leaf node labels [0,1,2,...]

plot = k3d.plot(grid_visible=True)

# Map semantic labels to colors: back=red, seat=green, leg=blue, armrest=yellow
color_map = {0: 0xff0000, 1: 0x00ff00, 2: 0x0000ff, 3: 0xffff00}

for i, obb in enumerate(genshape):
    label = labels_data[i]  # semantic label
    plot += make_box_k3d(obb, color=color_map.get(label, 0xffffff))

plot.display()

Output()